In [87]:
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_ollama import ChatOllama

In [88]:
llm = ChatOllama(model='llama3.1')
parser = StrOutputParser()
history = ChatMessageHistory()
history.add_user_message("My name is Sreehari")
history.add_ai_message("Hello Sreehari, I am ready to preside over your case.")

In [89]:
verdict_schema = {
    "title": "court_verdict",
    "type": "object",
    "properties": {
        "verdict": {"type": "string", "description": "The final decision"},
        "reasoning": {"type": "string", "description": "Why the judge chose this"}
    },
    "required": ["verdict", "reasoning"]
}

In [90]:
judge_llm = llm.with_structured_output(verdict_schema)

In [91]:
pro_prompt = ChatPromptTemplate.from_template("Topic: {topic}. Provide 1 strong argument FOR this. Be brief.")
con_prompt = ChatPromptTemplate.from_template("Topic: {topic}. Provide 1 strong argument AGAINST this. Be brief.")

In [92]:
pro_chain = pro_prompt | llm
con_chain = con_prompt | llm

In [99]:
full_court = (
    RunnablePassthrough.assign(history=lambda x:history.messages)
    | RunnablePassthrough.assign(topic=lambda x:x['topic'].strip().title())
    | RunnablePassthrough.assign(
        pro_arg = pro_chain,
        con_arg = con_chain
    )
    | {"verdict_data":ChatPromptTemplate.from_messages([
        ("system", "You are a Judge. Consider the history: {history}"),
        ("human", "Topic: {topic}\nPro: {pro_arg}\nCon: {con_arg}\nGive your verdict.")
    ]) | judge_llm,
      "original_topic": itemgetter("topic")
       }
)

In [100]:
result = full_court.invoke({"topic": "  artificial intelligence in healthcare  "})

In [101]:
result

{'verdict_data': {'verdict': '**The court finds that Artificial Intelligence (AI) in healthcare is a double-edged sword. While AI has the potential to improve diagnosis accuracy and reduce misdiagnoses, it also poses a risk of dehumanization of care, potentially leading to decreased patient satisfaction and outcomes. The integration of AI in healthcare must be carefully balanced with the need for human empathy and compassion in medical relationships.',
  'reasoning': 'The **Argument For** presents strong evidence that AI can enhance diagnosis accuracy by analyzing vast amounts of medical data, including images and patient histories. This can lead to better health outcomes, improved patient satisfaction, and reduced healthcare costs. However, the **Con** raises a crucial point about the potential for dehumanization of care when relying too heavily on technology. The court must weigh these competing interests and consider the balance between technological advancement and human touch in m